# DAY2 — 머신러닝용 데이터와 고정 분할 확인

배터리 한 개를 한 행으로 저장한 Parquet을 읽고, 관리 정보·특징·정답·제외 사유·CV 분할을 확인한다. 모델 학습은 다음 노트북에서 수행한다.

- Batch 1: 36개 = 개발 29개 + Hold-out 7개
- Batch 2: 테스트 39개
- Batch 3: 추가 검토 전 후보 44개
- 제외: Batch 1 수명 정답 불확실 10개, Batch 2/3 정답 누락 8/2개

가공 파일이 없다면 프로젝트 루트에서 `python -m src.day2_data_audit`, `python -m src.day2_dataset` 순서로 실행한다. 이 노트북은 저장된 표를 읽으며 분할을 다시 선택하지 않는다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/day2_dataset.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.day2_dataset import F1, F2, TARGET
DATA_DIR = ROOT / 'data/processed/day2'
TABLE_DIR = ROOT / 'outputs/tables/day2/dataset'
print('프로젝트:', ROOT)
print('커널:', sys.executable)

프로젝트: /Users/soobin/myfolder/workspace/mini-project-data-analysis
커널: /Users/soobin/myfolder/workspace/mini-project-data-analysis/.venv/bin/python


## 1. Parquet → pandas DataFrame

In [2]:
all_cells = pd.read_parquet(DATA_DIR / 'all_cells.parquet')
train = pd.read_parquet(DATA_DIR / 'train_batch1.parquet')
test = pd.read_parquet(DATA_DIR / 'test_batch2.parquet')
batch3_candidates = pd.read_parquet(DATA_DIR / 'batch3_candidates.parquet')
display(all_cells.groupby(['batch', 'partition'], observed=True).size().rename('항목 수').to_frame())
display(train[['batch', 'cell_id', 'barcode_key', 'charging_policy', *F2, TARGET, 'partition', 'cv_valid_fold']].head(8))

항목 수
batch  partition             
batch1 development         29
       excluded            10
       holdout              7
batch2 excluded             8
       test_batch2         39
batch3 batch3_candidate    44
       excluded             2

,batch,cell_id,barcode_key,charging_policy,log10_deltaQ_var,QD_slope_masked,cycle_life,partition,cv_valid_fold
0,batch1,5,EL150800464883,4.4C(80%)-4.4C,-4.178878,0.000002,1074.0,holdout,<NA>
1,batch1,6,EL150800463886,4.8C(80%)-4.8C,-3.768878,-0.000018,636.0,holdout,<NA>
2,batch1,7,EL150800465027,4.8C(80%)-4.8C,-3.813486,-0.000012,870.0,holdout,<NA>
3,batch1,9,EL150800460468,5.4C(40%)-3.6C,-4.059384,0.000005,1054.0,development,4
4,batch1,11,EL150800463882,5.4C(50%)-3C,-4.146897,0.000005,788.0,development,5
5,batch1,14,EL150800463838,5.4C(60%)-3C,-4.046070,0.000016,880.0,development,2
6,batch1,15,EL150800453113,5.4C(60%)-3C,-4.017949,0.000017,719.0,development,2
7,batch1,16,EL150800460653,5.4C(60%)-3.6C,-4.099414,0.000013,862.0,development,3


## 2. 자료형·결측·원래 스케일 확인

로그 변환은 행별 계산이다. 학습 데이터에서 추정하는 결측 대체·표준화는 아직 수행하지 않았다.

In [3]:
display(train.dtypes.to_frame('dtype'))
display(pd.DataFrame({'train 결측 수': train[[*F2, TARGET]].isna().sum(),
                      'test 결측 수': test[[*F2, TARGET]].isna().sum()}))
display(train[[*F2, TARGET]].describe())
assert train[F2].notna().all().all() and test[F2].notna().all().all()

,dtype
batch,string
cell_id,int64
barcode_key,string
channel_id_decoded,string
charging_policy,string
protocol_key,string
cycle_life,float64
deltaQ_var,float64
deltaQ_min,float64
deltaQ_status,string


,train 결측 수,test 결측 수
log10_deltaQ_var,0,0
QD_slope_masked,0,0
cycle_life,0,0


,log10_deltaQ_var,QD_slope_masked,cycle_life
count,36.000000,36.000000,36.000000
mean,-3.790902,-0.000020,788.416667
std,0.232827,0.000030,148.695734
min,-4.178878,-0.000149,534.000000
25%,-4.020434,-0.000023,681.000000
50%,-3.786476,-0.000016,772.500000
75%,-3.661912,-0.000007,871.500000
max,-3.350338,0.000017,1074.000000


## 3. 제외 사유 확인

제외 항목도 all_cells에 보존한다. 0.90 Ah를 일괄 적용하는 필터가 아니라 점검에서 특정한 Batch 1의 10개 기록을 명시적으로 구분한다.

In [4]:
exclusions = pd.read_csv(TABLE_DIR / 'exclusions.csv')
display(exclusions)
assert len(all_cells) == 139
assert len(train) == 36 and len(test) == 39
assert len(exclusions) == 20

,batch,cell_id,barcode_key,cycle_life,exclusion_reason
0,batch1,0,EL150800460514,1190.0,batch1_incomplete_lifetime_record
1,batch1,1,EL150800460486,1179.0,batch1_incomplete_lifetime_record
2,batch1,2,EL150800460623,1177.0,batch1_incomplete_lifetime_record
3,batch1,3,EL150800464977,1226.0,batch1_incomplete_lifetime_record
4,batch1,4,EL150800464865,1227.0,batch1_incomplete_lifetime_record
5,batch1,8,EL150800464898,879.0,batch1_incomplete_lifetime_record
6,batch1,10,EL150800463980,906.0,batch1_incomplete_lifetime_record
7,batch1,12,EL150800460472,902.0,batch1_incomplete_lifetime_record
8,batch1,13,EL150800460658,897.0,batch1_incomplete_lifetime_record
9,batch1,22,EL150800460433,891.0,batch1_incomplete_lifetime_record


## 4. 고정 Hold-out과 CV 목록

분할 키는 충전 프로토콜이며, seed 42와 20% 그룹 Hold-out을 사용했다. 같은 프로토콜은 학습·검증에 걸치지 않는다. Batch 2와 Hold-out은 개발 CV에 포함되지 않는다.

In [5]:
development = train.loc[train.partition.eq('development')].copy()
holdout = train.loc[train.partition.eq('holdout')].copy()
cv_manifest = pd.read_csv(TABLE_DIR / 'cv_manifest.csv')
display(pd.read_csv(TABLE_DIR / 'cv_summary.csv'))
print('개발:', len(development), 'Hold-out:', len(holdout), 'Test:', len(test))
assert not set(development.protocol_key) & set(holdout.protocol_key)
assert not set(train.barcode_key) & set(test.barcode_key)
assert holdout.cv_valid_fold.isna().all() and test.cv_valid_fold.isna().all()
for fold, frame in cv_manifest.groupby('fold'):
    assert not set(frame.loc[frame.role.eq('train'), 'protocol_key']) & set(frame.loc[frame.role.eq('valid'), 'protocol_key'])
display(train[['cell_id', 'protocol_key', 'partition', 'cv_valid_fold']].sort_values(['partition', 'cell_id']))

,fold,train_n,valid_n,train_protocols,valid_protocols
0,1,23,6,13,3
1,2,23,6,13,3
2,3,23,6,13,3
3,4,23,6,12,4
4,5,24,5,13,3


개발: 29 Hold-out: 7 Test: 39


,cell_id,protocol_key,partition,cv_valid_fold
3,9,5.4C(40%)-3.6C,development,4
4,11,5.4C(50%)-3C,development,5
5,14,5.4C(60%)-3C,development,2
6,15,5.4C(60%)-3C,development,2
7,16,5.4C(60%)-3.6C,development,3
8,17,5.4C(60%)-3.6C,development,3
9,18,5.4C(70%)-3C,development,1
10,19,5.4C(70%)-3C,development,1
11,20,5.4C(80%)-5.4C,development,5
12,21,5.4C(80%)-5.4C,development,5


## 5. 모델에 전달할 X와 y

기본 입력은 F2의 두 열만 선택한다. barcode·partition·수명 관련 플래그는 입력에 포함하지 않는다. 모델 선택은 개발 집합의 고정 CV에서 수행하며, 최종 설정을 정한 후 Batch 1 전체 재학습에 train을 사용한다.

In [6]:
X_dev, y_dev = development[F2].copy(), development[TARGET].copy()
X_holdout, y_holdout = holdout[F2].copy(), holdout[TARGET].copy()
X_test, y_test = test[F2].copy(), test[TARGET].copy()
X_batch1, y_batch1 = train[F2].copy(), train[TARGET].copy()
cv_splits = []
for fold in range(1, 6):
    validation = development.cv_valid_fold.eq(fold).to_numpy()
    import numpy as np
    cv_splits.append((np.flatnonzero(~validation), np.flatnonzero(validation)))
print('F1:', F1, 'F2:', F2)
print('X_dev:', X_dev.shape, 'X_holdout:', X_holdout.shape, 'X_test:', X_test.shape)
print('CV 회차:', len(cv_splits))
display(X_dev.head())

F1: ['log10_deltaQ_var'] F2: ['log10_deltaQ_var', 'QD_slope_masked']
X_dev: (29, 2) X_holdout: (7, 2) X_test: (39, 2)
CV 회차: 5


,log10_deltaQ_var,QD_slope_masked
3,-4.059384,0.000005
4,-4.146897,0.000005
5,-4.046070,0.000016
6,-4.017949,0.000017
7,-4.099414,0.000013


## 근거와 다음 단계

[학습 테이블 구성 보고서](../outputs/reports/day2/dataset_preparation_report.md) · [학습 전 데이터 점검](../outputs/reports/day2/data_audit_report.md) · [가공 코드](../src/day2_dataset.py)

다음 단계는 같은 CV 분할을 사용한 중앙값 기준 모델과 F1/F2 선형 회귀·Ridge 비교다. Batch 3는 곡선 품질 추가 점검 전까지 후보 표로만 유지한다.